# VadCLIP Class Prompt Alignment Diagnostics (Post-Train)

Notebook này **không** dùng để chọn prompt. Nó phân tích **sau fine-tune** một class cụ thể (mặc định `Explosion`):

- Model tụt vì anomaly/positive frames giảm score, hay background/negative frames tăng score?
- Anomaly frames đang align với prompt nào / class nào?
- Prompt nào kích hoạt đúng ở anomaly frames và prompt nào gây false alarm trên background?

**Luồng đề xuất:**

1. `analyze_vadclip_prompt_text_space.ipynb` → chấm điểm + sinh Top-1 / diversity Top-K weighted bank  
2. Train multi-prompt với bank đó  
3. **Notebook này** → gắn `PROMPT_JSON` / `PROMPT_WEIGHTS_JSON` / checkpoint vừa train, đổi `TARGET_CLASS` nếu cần  

Notebook tái sử dụng được: chỉ cần đổi checkpoint, prompt JSON, prompt weights và `TARGET_CLASS`. Kết quả lưu dưới `code/results/analyze/` (hoặc `Result/` tùy cấu hình cell).


In [ ]:
from pathlib import Path
import csv
import json
import math
import shutil
import subprocess
import sys

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
else:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR

DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]

SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_ROOT = PROJECT_ROOT / 'code' / 'results'
EVALUATE_RESULTS_DIR = RESULTS_ROOT / 'evaluate'
ANALYZE_RESULTS_DIR = RESULTS_ROOT / 'analyze'
for result_dir in [RESULTS_ROOT, EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR]:
    result_dir.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_DIR:', SRC_DIR)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('ANALYZE_RESULTS_DIR:', ANALYZE_RESULTS_DIR)


## 1. Config

Đổi các biến trong cell này để phân tích model/class khác. Mặc định đang phân tích `Explosion` với weighted prompt output từ notebook analyze.


In [ ]:
# Class cần phân tích. Có thể đổi sang Arson, Assault, Shoplifting, ...
TARGET_CLASS = 'Explosion'

# Model cần phân tích.
# Nếu muốn phân tích checkpoint tốt nhất theo mAP, đổi path sang epoch checkpoint tương ứng,
# ví dụ: VadCLIP/src/model/epoch_checkpoints_multi_prompt_.../model_epoch_004_multi_prompt.pth
MODEL_TYPE = 'multi_prompt'  # currently supports: multi_prompt
MODEL_PATH = PROJECT_ROOT / 'VadCLIP' / 'src' / 'model' / 'model_ucf_multi_prompt_manual_caption_analyze_prompt_weighted_no_regularize_scratch.pth'

# Baseline VadCLIP checkpoint để so sánh score drift.
BASELINE_MODEL_PATH = PROJECT_ROOT / 'model_ucf.pth'

# Prompt config của model fine-tune.
PROMPT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'
PROMPT_WEIGHTS_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'  # set None nếu dùng uniform prompt
PROMPT_MODE = 'manual_caption'

# Data/test protocol.
# Prefer local /content features. If the Drive folder is missing, the next cell can extract UCFClipFeatures.zip/tar from Drive.
FEATURE_ROOT = LOCAL_FEATURE_ROOT if IN_COLAB and LOCAL_FEATURE_ROOT.exists() else DRIVE_FEATURE_ROOT
TEST_LIST = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv'
GT_PATH = PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy'

# Output.
OUTPUT_DIR = ANALYZE_RESULTS_DIR / f'class_prompt_alignment_{TARGET_CLASS.lower()}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('TARGET_CLASS:', TARGET_CLASS)
print('MODEL_PATH:', MODEL_PATH)
print('BASELINE_MODEL_PATH:', BASELINE_MODEL_PATH)
print('PROMPT_JSON:', PROMPT_JSON)
print('PROMPT_WEIGHTS_JSON:', PROMPT_WEIGHTS_JSON)
print('FEATURE_ROOT:', FEATURE_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

for required_path in [MODEL_PATH, BASELINE_MODEL_PATH, PROMPT_JSON, TEST_LIST, GT_PATH]:
    if required_path is not None and not Path(required_path).exists():
        raise FileNotFoundError(required_path)


## 1b. Chuẩn Bị Feature Root

Cell này tự động dùng `/content/UCFClipFeatures` nếu đã có. Nếu chưa có và Drive không có folder feature, nó sẽ tìm `UCFClipFeatures.zip`, `.tar`, `.tar.gz`, hoặc `.tgz` trong Drive, copy sang `/content`, giải nén, rồi dùng local feature root.


In [ ]:
import time


def prepare_feature_root():
    global FEATURE_ROOT

    if FEATURE_ROOT.exists():
        print('Using existing feature root:', FEATURE_ROOT)
        return FEATURE_ROOT

    if not IN_COLAB:
        raise FileNotFoundError(
            f'Missing feature root: {FEATURE_ROOT}. On local machine, create UCFClipFeatures/ or update FEATURE_ROOT.'
        )

    if LOCAL_FEATURE_ROOT.exists():
        FEATURE_ROOT = LOCAL_FEATURE_ROOT
        print('Using local feature root:', FEATURE_ROOT)
        return FEATURE_ROOT

    archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
    if archive is None:
        archive_names = ', '.join(path.name for path in DRIVE_FEATURE_ARCHIVES)
        raise FileNotFoundError(
            f'Missing UCFClipFeatures folder and no archive found. Expected one of: {archive_names}'
        )

    print('Feature folder missing. Extracting archive:', archive)
    start = time.time()
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copying archive to local runtime:', local_archive)
        shutil.copy2(archive, local_archive)
    else:
        print('Local archive already exists:', local_archive)

    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)

    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive must contain top-level folder UCFClipFeatures/')

    FEATURE_ROOT = LOCAL_FEATURE_ROOT
    print(f'Prepared local features in {(time.time() - start) / 60:.1f} minutes')
    print('Feature root:', FEATURE_ROOT)
    return FEATURE_ROOT


FEATURE_ROOT = prepare_feature_root()


## 2. Imports And Helpers


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader

from model import CLIPVAD
from model_multi_prompt import CLIPVADMultiPrompt
from ucf_evaluate import UCFRelativeTestDataset
from utils.prompt_replacement import build_multi_prompt_groups, build_multi_prompt_weights
from utils.tools import get_batch_mask, get_prompt_text

LABEL_MAP = {
    'Normal': 'Normal',
    'Abuse': 'Abuse',
    'Arrest': 'Arrest',
    'Arson': 'Arson',
    'Assault': 'Assault',
    'Burglary': 'Burglary',
    'Explosion': 'Explosion',
    'Fighting': 'Fighting',
    'RoadAccidents': 'RoadAccidents',
    'Robbery': 'Robbery',
    'Shooting': 'Shooting',
    'Shoplifting': 'Shoplifting',
    'Stealing': 'Stealing',
    'Vandalism': 'Vandalism',
}

PROMPT_LABEL_MAP = {
    'Normal': 'normal',
    'Abuse': 'abuse',
    'Arrest': 'arrest',
    'Arson': 'arson',
    'Assault': 'assault',
    'Burglary': 'burglary',
    'Explosion': 'explosion',
    'Fighting': 'fighting',
    'RoadAccidents': 'roadAccidents',
    'Robbery': 'robbery',
    'Shooting': 'shooting',
    'Shoplifting': 'shoplifting',
    'Stealing': 'stealing',
    'Vandalism': 'vandalism',
}

CLASS_NAMES = list(PROMPT_LABEL_MAP.keys())
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VISUAL_LENGTH = 256

MODEL_CONFIG = dict(
    num_class=14,
    embed_dim=512,
    visual_length=256,
    visual_width=512,
    visual_head=1,
    visual_layers=2,
    attn_window=8,
    prompt_prefix=10,
    prompt_postfix=10,
)


def build_lengths(length, visual_length=VISUAL_LENGTH):
    length_remaining = int(length)
    lengths = torch.zeros(int(length / visual_length) + 1)
    for j in range(int(length / visual_length) + 1):
        if j == 0 and length_remaining < visual_length:
            lengths[j] = length_remaining
        elif j == 0 and length_remaining > visual_length:
            lengths[j] = visual_length
            length_remaining -= visual_length
        elif length_remaining > visual_length:
            lengths[j] = visual_length
            length_remaining -= visual_length
        else:
            lengths[j] = length_remaining
    return lengths.to(int)


def load_model(model_cls, model_path):
    model = model_cls(**MODEL_CONFIG, device=DEVICE)
    state = torch.load(model_path, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(state, strict=False)
    model.to(DEVICE).eval()
    print(f'Loaded {model_cls.__name__}:', model_path)
    print('  missing:', len(missing), '| unexpected:', len(unexpected))
    return model


def safe_auc_ap(gt, scores):
    gt = np.asarray(gt)
    scores = np.asarray(scores)
    if len(np.unique(gt)) < 2:
        return math.nan, math.nan
    return float(roc_auc_score(gt, scores) * 100), float(average_precision_score(gt, scores) * 100)


def repeat_clip_to_frame(x):
    return np.repeat(np.asarray(x), 16)




def prepare_visual_batch(visual, total_length, visual_length=VISUAL_LENGTH):
    """Return feature chunks [N, visual_length, 512] and valid lengths [N]."""
    if not torch.is_tensor(visual):
        visual = torch.tensor(visual)
    visual = visual.float().cpu()
    total_length = int(total_length)

    if visual.ndim == 2:
        visual_batch = visual.unsqueeze(0)
    elif visual.ndim == 3:
        visual_batch = visual
    else:
        raise ValueError(f'Expected visual feature with 2 or 3 dims, got shape {tuple(visual.shape)}')

    lengths = build_lengths(total_length, visual_length=visual_length)
    needed_chunks = int(len(lengths))

    if visual_batch.shape[0] > needed_chunks:
        visual_batch = visual_batch[:needed_chunks]
    elif visual_batch.shape[0] < needed_chunks:
        pad_shape = (needed_chunks - visual_batch.shape[0], visual_length, visual_batch.shape[-1])
        visual_batch = torch.cat([visual_batch, torch.zeros(pad_shape, dtype=visual_batch.dtype)], dim=0)

    if visual_batch.shape[1] != visual_length:
        raise ValueError(
            f'Feature chunk length must be {visual_length}, got shape {tuple(visual_batch.shape)}. '
            'Create dataset with UCFRelativeTestDataset(VISUAL_LENGTH, ...).'
        )

    return visual_batch.to(DEVICE), lengths.to(DEVICE)


def run_baseline_full_sequence(model, visual, total_length, prompt_text):
    visual_batch, lengths = prepare_visual_batch(visual, total_length)
    padding_mask = get_batch_mask(lengths, VISUAL_LENGTH).to(DEVICE)
    _, logits1, logits2 = model(visual_batch, padding_mask, prompt_text, lengths)
    logits1 = logits1.reshape(-1, logits1.shape[-1])[:total_length]
    logits2 = logits2.reshape(-1, logits2.shape[-1])[:total_length]
    return logits1, logits2


def run_multi_prompt_full_sequence(model, visual, total_length, prompt_groups, prompt_weights=None):
    visual_batch, lengths = prepare_visual_batch(visual, total_length)
    padding_mask = get_batch_mask(lengths, VISUAL_LENGTH).to(DEVICE)
    _, logits1, logits2, prompt_logits = model(
        visual_batch,
        padding_mask,
        prompt_groups,
        lengths,
        return_prompt_logits=True,
        prompt_weights=prompt_weights,
    )
    logits1 = logits1.reshape(-1, logits1.shape[-1])[:total_length]
    logits2 = logits2.reshape(-1, logits2.shape[-1])[:total_length]
    prompt_logits = prompt_logits.reshape(-1, prompt_logits.shape[-2], prompt_logits.shape[-1])[:total_length]
    return logits1, logits2, prompt_logits

def save_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8')
    print('Saved:', path)


def save_plot(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    print('Saved:', path)


## 3. Load Dataset, Models, Prompts


In [ ]:
dataset = UCFRelativeTestDataset(VISUAL_LENGTH, TEST_LIST, FEATURE_ROOT)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
gt_all = np.load(GT_PATH)

baseline_model = load_model(CLIPVAD, BASELINE_MODEL_PATH)
model = load_model(CLIPVADMultiPrompt, MODEL_PATH)

baseline_prompt_text = get_prompt_text(LABEL_MAP)
prompt_groups = build_multi_prompt_groups(
    str(PROMPT_JSON),
    PROMPT_LABEL_MAP,
    'manual',
    prompt_mode=PROMPT_MODE,
)
prompt_weights = build_multi_prompt_weights(str(PROMPT_WEIGHTS_JSON), PROMPT_LABEL_MAP, DEVICE) if PROMPT_WEIGHTS_JSON else None

print('Prompt groups:', len(prompt_groups), 'classes x', len(prompt_groups[0]), 'prompts')
print('Prompt weights:', None if prompt_weights is None else tuple(prompt_weights.shape))
for i, prompt in enumerate(prompt_groups[CLASS_TO_INDEX[TARGET_CLASS]]):
    weight_text = '' if prompt_weights is None else f' weight={float(prompt_weights[CLASS_TO_INDEX[TARGET_CLASS], i]):.4f}'
    print(f'{TARGET_CLASS}#{i}{weight_text}: {prompt}')


## 4. Inference With Prompt Logits

Cell này chạy inference trên toàn test set, nhưng chỉ lưu chi tiết cho từng video thuộc `TARGET_CLASS`. Với model multi-prompt, notebook lấy thêm `prompt_logits [T, 14, K]` để phân tích frame-prompt alignment.


In [ ]:
target_records = []
gt_cursor = 0

with torch.no_grad():
    for item in dataloader:
        visual = item[0].squeeze(0)
        label = item[1][0] if isinstance(item[1], (list, tuple)) else item[1]
        length = int(item[2])
        video_id = item[3][0] if isinstance(item[3], (list, tuple)) else item[3]
        len_cur = length

        frame_count = int(len_cur * 16)
        gt_frame = gt_all[gt_cursor: gt_cursor + frame_count]
        gt_cursor += frame_count

        if label != TARGET_CLASS:
            continue

        base_logits1, base_logits2 = run_baseline_full_sequence(
            baseline_model,
            visual,
            len_cur,
            baseline_prompt_text,
        )
        base_logits1 = base_logits1[:len_cur]
        base_logits2 = base_logits2[:len_cur]
        base_classifier_clip = torch.sigmoid(base_logits1.squeeze(-1)).detach().cpu().numpy()
        base_alignment_clip = (1 - base_logits2.softmax(dim=-1)[:, 0]).detach().cpu().numpy()

        logits1, logits2, prompt_logits = run_multi_prompt_full_sequence(
            model,
            visual,
            len_cur,
            prompt_groups,
            prompt_weights=prompt_weights,
        )
        logits1 = logits1[:len_cur]
        logits2 = logits2[:len_cur]
        prompt_logits = prompt_logits[:len_cur]

        classifier_clip = torch.sigmoid(logits1.squeeze(-1)).detach().cpu().numpy()
        alignment_clip = (1 - logits2.softmax(dim=-1)[:, 0]).detach().cpu().numpy()
        prompt_logits_np = prompt_logits.detach().cpu().numpy()

        target_records.append({
            'video_id': video_id,
            'label': label,
            'length_clip': len_cur,
            'gt_frame': gt_frame.astype(np.float32),
            'gt_clip': gt_frame.reshape(-1, 16).max(axis=1).astype(np.float32)[:len_cur],
            'baseline_classifier_clip': base_classifier_clip,
            'baseline_alignment_clip': base_alignment_clip,
            'classifier_clip': classifier_clip,
            'alignment_clip': alignment_clip,
            'prompt_logits': prompt_logits_np,
        })

print('Target videos:', len(target_records))
print('GT cursor/frame length:', gt_cursor, '/', len(gt_all))
if gt_cursor != len(gt_all):
    raise ValueError(f'GT cursor mismatch: {gt_cursor} vs {len(gt_all)}')


## 5. Score Drift: Positive vs Negative Frames

Bảng này cho biết class đang tụt do positive/anomaly frames giảm score hay negative/background frames tăng score.


In [ ]:
video_rows = []
for rec in target_records:
    gt_clip = rec['gt_clip'].astype(bool)
    gt_frame = rec['gt_frame'].astype(bool)
    base_cls_frame = repeat_clip_to_frame(rec['baseline_classifier_clip'])[:len(gt_frame)]
    cls_frame = repeat_clip_to_frame(rec['classifier_clip'])[:len(gt_frame)]
    base_align_frame = repeat_clip_to_frame(rec['baseline_alignment_clip'])[:len(gt_frame)]
    align_frame = repeat_clip_to_frame(rec['alignment_clip'])[:len(gt_frame)]

    row = {
        'video_id': rec['video_id'],
        'class_name': rec['label'],
        'num_clip': rec['length_clip'],
        'num_frames': len(gt_frame),
        'positive_frames': int(gt_frame.sum()),
    }
    for branch, base_scores, scores in [
        ('classifier', base_cls_frame, cls_frame),
        ('alignment', base_align_frame, align_frame),
    ]:
        auc_base, ap_base = safe_auc_ap(gt_frame, base_scores)
        auc_new, ap_new = safe_auc_ap(gt_frame, scores)
        row[f'baseline_{branch}_auc'] = auc_base
        row[f'new_{branch}_auc'] = auc_new
        row[f'delta_{branch}_auc'] = auc_new - auc_base if not math.isnan(auc_base) and not math.isnan(auc_new) else math.nan
        row[f'baseline_{branch}_ap'] = ap_base
        row[f'new_{branch}_ap'] = ap_new
        row[f'delta_{branch}_ap'] = ap_new - ap_base if not math.isnan(ap_base) and not math.isnan(ap_new) else math.nan
        for frame_type, mask in [('positive', gt_frame), ('negative', ~gt_frame)]:
            if mask.sum() == 0:
                row[f'baseline_{branch}_{frame_type}_mean'] = math.nan
                row[f'new_{branch}_{frame_type}_mean'] = math.nan
                row[f'delta_{branch}_{frame_type}_mean'] = math.nan
            else:
                base_mean = float(base_scores[mask].mean())
                new_mean = float(scores[mask].mean())
                row[f'baseline_{branch}_{frame_type}_mean'] = base_mean
                row[f'new_{branch}_{frame_type}_mean'] = new_mean
                row[f'delta_{branch}_{frame_type}_mean'] = new_mean - base_mean
        row[f'baseline_{branch}_margin'] = row[f'baseline_{branch}_positive_mean'] - row[f'baseline_{branch}_negative_mean']
        row[f'new_{branch}_margin'] = row[f'new_{branch}_positive_mean'] - row[f'new_{branch}_negative_mean']
        row[f'delta_{branch}_margin'] = row[f'new_{branch}_margin'] - row[f'baseline_{branch}_margin']
    video_rows.append(row)

video_df = pd.DataFrame(video_rows)
save_df(video_df, OUTPUT_DIR / f'{TARGET_CLASS.lower()}_video_score_drift.csv')

summary_cols = [
    'delta_classifier_auc', 'delta_classifier_positive_mean', 'delta_classifier_negative_mean', 'delta_classifier_margin',
    'delta_alignment_auc', 'delta_alignment_positive_mean', 'delta_alignment_negative_mean', 'delta_alignment_margin',
]
print('Mean deltas for', TARGET_CLASS)
print(video_df[summary_cols].mean(numeric_only=True).to_string())
print('
Worst videos by classifier AUC delta:')
print(video_df.sort_values('delta_classifier_auc')[['video_id','positive_frames','delta_classifier_auc','delta_classifier_positive_mean','delta_classifier_negative_mean','delta_classifier_margin']].head(10).to_string(index=False))


## 6. Frame-Prompt Alignment Summary

Bảng này cho biết positive frames và negative frames của target class align mạnh với prompt nào. Nếu positive frames của `Explosion` align với `Arson` hoặc `Assault`, đó là dấu hiệu prompt confusion. Nếu negative frames align mạnh với target prompt, đó là dấu hiệu false alarm background.


In [ ]:
alignment_rows = []
class_prompt_labels = []
for class_index, class_name in enumerate(CLASS_NAMES):
    for prompt_index, prompt in enumerate(prompt_groups[class_index]):
        class_prompt_labels.append((class_index, class_name, prompt_index, prompt))

for rec in target_records:
    prompt_logits = rec['prompt_logits']
    gt_clip = rec['gt_clip'].astype(bool)
    for frame_type, mask in [('positive', gt_clip), ('negative', ~gt_clip), ('all', np.ones_like(gt_clip, dtype=bool))]:
        if mask.sum() == 0:
            continue
        mean_logits = prompt_logits[mask].mean(axis=0)  # [14,K]
        flat_order = np.argsort(mean_logits.reshape(-1))[::-1]
        target_idx = CLASS_TO_INDEX[TARGET_CLASS]
        target_best_prompt = int(np.argmax(mean_logits[target_idx]))
        target_best_score = float(mean_logits[target_idx, target_best_prompt])
        for rank, flat_idx in enumerate(flat_order[:10], start=1):
            c = int(flat_idx // mean_logits.shape[1])
            k = int(flat_idx % mean_logits.shape[1])
            alignment_rows.append({
                'video_id': rec['video_id'],
                'frame_type': frame_type,
                'rank': rank,
                'class_index': c,
                'class_name': CLASS_NAMES[c],
                'prompt_index': k,
                'prompt_text': prompt_groups[c][k],
                'mean_prompt_logit': float(mean_logits[c, k]),
                'target_best_prompt_index': target_best_prompt,
                'target_best_prompt_text': prompt_groups[target_idx][target_best_prompt],
                'target_best_score': target_best_score,
                'best_minus_target': float(mean_logits[c, k] - target_best_score),
                'is_target_class': c == target_idx,
            })

alignment_df = pd.DataFrame(alignment_rows)
save_df(alignment_df, OUTPUT_DIR / f'{TARGET_CLASS.lower()}_top_prompt_alignment_by_video.csv')

print('Top class/prompt counts on positive frames:')
pos_top = alignment_df[(alignment_df['frame_type']=='positive') & (alignment_df['rank']==1)]
print(pos_top.groupby(['class_name','prompt_index','prompt_text']).size().sort_values(ascending=False).head(20).to_string())

print('
Top class/prompt counts on negative frames:')
neg_top = alignment_df[(alignment_df['frame_type']=='negative') & (alignment_df['rank']==1)]
print(neg_top.groupby(['class_name','prompt_index','prompt_text']).size().sort_values(ascending=False).head(20).to_string())


## 7. Aggregate Prompt Behavior

Tổng hợp trên toàn bộ video của target class: prompt nào có mean cao nhất trên positive frames, negative frames, và margin positive-negative.


In [ ]:
prompt_behavior_rows = []
all_prompt_logits = []
all_gt_clip = []
for rec in target_records:
    all_prompt_logits.append(rec['prompt_logits'])
    all_gt_clip.append(rec['gt_clip'].astype(bool))
all_prompt_logits = np.concatenate(all_prompt_logits, axis=0)
all_gt_clip = np.concatenate(all_gt_clip, axis=0)

for c, class_name, k, prompt in class_prompt_labels:
    values = all_prompt_logits[:, c, k]
    pos_values = values[all_gt_clip]
    neg_values = values[~all_gt_clip]
    row = {
        'class_index': c,
        'class_name': class_name,
        'prompt_index': k,
        'prompt_text': prompt,
        'positive_mean_logit': float(pos_values.mean()) if len(pos_values) else math.nan,
        'negative_mean_logit': float(neg_values.mean()) if len(neg_values) else math.nan,
        'margin_pos_minus_neg': float(pos_values.mean() - neg_values.mean()) if len(pos_values) and len(neg_values) else math.nan,
        'positive_p95_logit': float(np.percentile(pos_values, 95)) if len(pos_values) else math.nan,
        'negative_p95_logit': float(np.percentile(neg_values, 95)) if len(neg_values) else math.nan,
    }
    prompt_behavior_rows.append(row)

prompt_behavior_df = pd.DataFrame(prompt_behavior_rows)
save_df(prompt_behavior_df, OUTPUT_DIR / f'{TARGET_CLASS.lower()}_prompt_behavior_global.csv')

print('Top prompts by positive mean:')
print(prompt_behavior_df.sort_values('positive_mean_logit', ascending=False).head(15)[['class_name','prompt_index','positive_mean_logit','negative_mean_logit','margin_pos_minus_neg','prompt_text']].to_string(index=False))
print('
Top prompts by margin:')
print(prompt_behavior_df.sort_values('margin_pos_minus_neg', ascending=False).head(15)[['class_name','prompt_index','positive_mean_logit','negative_mean_logit','margin_pos_minus_neg','prompt_text']].to_string(index=False))
print('
Worst target-class prompts by margin:')
print(prompt_behavior_df[prompt_behavior_df['class_name']==TARGET_CLASS].sort_values('margin_pos_minus_neg')[['class_name','prompt_index','positive_mean_logit','negative_mean_logit','margin_pos_minus_neg','prompt_text']].to_string(index=False))


## 8. Timeline Plots For Worst Videos

Vẽ timeline cho các video target class bị tụt nhiều nhất. Mỗi hình gồm ground truth, baseline/new classifier score, baseline/new alignment score, target prompt logit tốt nhất và best-other prompt logit.


In [ ]:
timeline_dir = OUTPUT_DIR / 'timelines'
timeline_dir.mkdir(parents=True, exist_ok=True)
worst_video_ids = video_df.sort_values('delta_classifier_auc')['video_id'].head(8).tolist()

for video_id in worst_video_ids:
    rec = next(r for r in target_records if r['video_id'] == video_id)
    gt_clip = rec['gt_clip'].astype(float)
    x = np.arange(rec['length_clip'])
    prompt_logits = rec['prompt_logits']
    target_idx = CLASS_TO_INDEX[TARGET_CLASS]
    target_best = prompt_logits[:, target_idx, :].max(axis=1)
    other_logits = prompt_logits.copy()
    other_logits[:, target_idx, :] = -1e9
    other_best_flat = other_logits.reshape(other_logits.shape[0], -1).argmax(axis=1)
    other_best_score = other_logits.reshape(other_logits.shape[0], -1).max(axis=1)
    other_class_idx = other_best_flat // prompt_logits.shape[2]
    top_other_class = pd.Series([CLASS_NAMES[int(i)] for i in other_class_idx]).mode().iloc[0]

    plt.figure(figsize=(14, 7))
    plt.plot(x, rec['baseline_classifier_clip'], label='baseline classifier', alpha=0.85)
    plt.plot(x, rec['classifier_clip'], label='new classifier', alpha=0.85)
    plt.plot(x, rec['baseline_alignment_clip'], label='baseline alignment', alpha=0.65)
    plt.plot(x, rec['alignment_clip'], label='new alignment', alpha=0.65)
    plt.plot(x, target_best, label=f'{TARGET_CLASS} best prompt logit', alpha=0.75)
    plt.plot(x, other_best_score, label=f'best other prompt logit, mode={top_other_class}', alpha=0.75)
    ymin, ymax = plt.ylim()
    plt.fill_between(x, ymin, ymax, where=gt_clip.astype(bool), color='red', alpha=0.12, label='GT anomaly')
    plt.title(f'{TARGET_CLASS} timeline: {video_id}')
    plt.xlabel('clip timestep')
    plt.ylabel('score / logit')
    plt.legend(loc='best', fontsize=8)
    save_plot(timeline_dir / f'{TARGET_CLASS}_{video_id}.png')

print('Timeline videos:', worst_video_ids)


## 9. Save Summary JSON


In [ ]:
summary = {
    'target_class': TARGET_CLASS,
    'model_path': str(MODEL_PATH),
    'baseline_model_path': str(BASELINE_MODEL_PATH),
    'prompt_json': str(PROMPT_JSON),
    'prompt_weights_json': None if PROMPT_WEIGHTS_JSON is None else str(PROMPT_WEIGHTS_JSON),
    'prompt_mode': PROMPT_MODE,
    'num_target_videos': len(target_records),
    'score_drift_mean': video_df[summary_cols].mean(numeric_only=True).to_dict(),
    'worst_classifier_auc_videos': video_df.sort_values('delta_classifier_auc')[['video_id','delta_classifier_auc','delta_classifier_positive_mean','delta_classifier_negative_mean','delta_classifier_margin']].head(10).to_dict(orient='records'),
    'top_positive_prompt_counts': pos_top.groupby(['class_name','prompt_index','prompt_text']).size().sort_values(ascending=False).head(20).reset_index(name='count').to_dict(orient='records'),
    'top_negative_prompt_counts': neg_top.groupby(['class_name','prompt_index','prompt_text']).size().sort_values(ascending=False).head(20).reset_index(name='count').to_dict(orient='records'),
}
summary_path = OUTPUT_DIR / f'{TARGET_CLASS.lower()}_diagnostics_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])
